# Ma Sói — Behavior Cloning trên Colab

Train `Learned Policy V0` từ dataset self-play của bot heuristic (BOT_SELF_LEARNING §17).

## Trước khi mở notebook này

Ở máy bạn, dataset phải đã được sinh và ĐÃ QUA leak validator:

```powershell
npx tsx apps/server/scripts/selfplay.ts --games 10000 --players 8 --preset --defense --seed bc --trajectories .tmp/bc --trace-games 10000 --no-jitter --quiet
npm run ai:validate-dataset -- .tmp/bc/trajectories.jsonl
npm run ai:encode -- --in .tmp/bc/trajectories.jsonl --out .tmp/bc/enc
```

Validator phải in `dataset SẠCH`. `--no-jitter` là bắt buộc: thiếu nó thì teacher không
tất định, trần độ khớp tụt về khoảng 0,60 và mọi con số phía sau đọc sai.

Rồi đóng gói và tải lên Google Drive:

```powershell
Compress-Archive -Path ai-training\masoi_training, .tmp\bc\enc -DestinationPath .tmp\bc\bc-package.zip -Force
```

Zip gồm `masoi_training/` (mã train) và `enc/` (tensor `.bin`). Trajectory JSONL 6 GB
KHÔNG cần tải lên — tensor đã encode nén còn khoảng 84 MB vì phần lớn đặc trưng là one-hot.

**Ranh giới không đổi (§39):** luật game và observation encoder ở TypeScript, chạy dưới
máy bạn. Colab chỉ nhận số và train — nó không parse trajectory, không biết vai nào là Sói.

## 0. Kiểm GPU

Colab đôi khi cấp runtime CPU mà không nói gì. Train vẫn chạy, chỉ chậm hơn 3–4 lần — và
bạn sẽ không biết vì sao. Cell này nói thẳng ra.

Không có GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# torch ban moi tach exporter ONNX ra goi rieng. Thieu no thi train van xong nhung
# metrics.json ghi `onnxError: No module named 'onnxscript'` va khong co model.onnx.
# Cai o day chu khong o cell train: train_bc chay trong tien trinh con, nen goi vua
# cai co hieu luc ngay ma khong can restart runtime.
%pip install -q onnxscript

import torch

print("torch", torch.__version__, "| cuda build", torch.version.cuda)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  {name}  ({vram:.1f} GB)")
else:
    print("KHONG CO GPU - Runtime > Change runtime type > T4 GPU, roi chay lai cell nay.")


## 1. Nạp dữ liệu từ Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Sửa cho khớp chỗ bạn đặt file trên Drive.
ZIP = "/content/drive/MyDrive/bc-package.zip"

!rm -rf /content/bc && mkdir -p /content/bc
!unzip -q "$ZIP" -d /content/bc
!ls /content/bc /content/bc/enc

## 2. Kiểm dữ liệu trước khi train

Ba câu hỏi, hỏi trước khi tiêu một lượt GPU:

1. File có khớp `meta.json` không — một file cụt vẫn reshape "thành công" và làm lệch
   mọi nhãn đi một hàng.
2. Ba phần train/val/test có cộng lại đúng bằng cả tập không (§15).
3. **Mọi nhãn có phải một nước HỢP LỆ theo chính mask của nó không** — nếu không,
   dataset đang dạy model đi nước ngoài luật.

In [ ]:
import gc
import glob
import os
import sys

sys.path.insert(0, "/content/bc")
from masoi_training.data import action_distribution, load

# Tu tim thu muc dataset thay vi hard-code ten. Goi day du dat ten `enc`, goi slim
# (bo scores.f32.bin) dat ten `enc-slim`; hard-code mot cai thi cai kia hong o day
# voi mot loi khong noi len nguyen nhan.
found = [p for p in sorted(glob.glob("/content/bc/*")) if os.path.isfile(os.path.join(p, "meta.json"))]
if len(found) != 1:
    raise SystemExit(f"can dung 1 thu muc co meta.json trong /content/bc, thay: {found}")
DATA = found[0]
print("dataset dir:", DATA)


def check(path):
    """Kiem trong mot ham, de moi mang numpy chet ngay khi ham tra ve.

    Day KHONG phai chuyen phong cach. `load()` nap TOAN BO tensor vao RAM
    (~3,1 GB voi goi day du). Neu de `data` song o pham vi notebook thi khi cell
    train spawn tien trinh con - tien trinh do nap MOT BAN NUA, cong ba split -
    tong vuot RAM cua Colab free va tien trinh con bi giet voi exit -9 (SIGKILL),
    log trong. Chinh no da xay ra. Tra ve so, khong tra ve mang.
    """
    data = load(path)  # nem ngay neu file khong khop meta.json

    sizes = {name: len(data.split(name)) for name in ("train", "validation", "test")}
    assert sum(sizes.values()) == len(data)
    assert data.masks[range(len(data)), data.actions].all(), "co nhan tro vao hanh dong BAT HOP LE"
    assert set(data.rewards.tolist()) <= {-1.0, 1.0}

    # `optimal.u8.bin` la thu `agreementTieAware` dua vao. Thieu no thi chi so
    # chinh cua §17 se la null va ban chi con `agreement` - thuoc cham oan moi
    # nuoc hoa diem.
    assert data.optimal is not None, "thieu optimal.u8.bin - chay lai ai:encode"

    return {
        "version": data.meta.get("datasetVersion"),
        "commit": (data.meta.get("gitCommit") or "")[:8],
        "rows": len(data),
        "obs": data.obs_size,
        "actions": data.action_size,
        "games": data.meta.get("games"),
        "sizes": sizes,
        "classes": len(action_distribution(data)),
        "scores": data.scores is not None,
    }


info = check(DATA)
gc.collect()

print("dataset  ", info["version"], "commit", info["commit"])
print("rows     ", info["rows"], "| obs", info["obs"], "| actions", info["actions"])
print("games    ", info["games"])
print("split    ", info["sizes"])
print("lop hanh dong (§43):", info["classes"])
print("scores   ", "co (chi dung khi --distill-alpha > 0)" if info["scores"] else "khong - mac dinh khong can")
print("\nOK - train duoc. RAM da duoc tra lai truoc khi sang cell train.")


## 3. Chạy thử 2 epoch

Trước khi chạy thật. Một lỗi cấu hình lộ ra ở đây mất 30 giây, lộ ra ở cell sau mất cả
lượt GPU.

In [ ]:
import subprocess
import sys

done = subprocess.run(
    [sys.executable, "-m", "masoi_training.train_bc",
     "--data", DATA, "--out", "/content/smoke", "--epochs", "2"],
    cwd="/content/bc",
)
if done.returncode != 0:
    raise SystemExit(f"smoke test hong (exit {done.returncode}) - doc log phia tren")
print("smoke OK")


## 4. Train thật

`--batch-size 512` là mặc định của script và là con số mọi báo cáo khác trong repo dùng,
nên giữ nó thì `metrics.json` so được với các lần chạy cũ. Muốn nhanh hơn thì `1024`:
ít step hơn nên ít overhead Python hơn, đổi lại gradient thô hơn một chút.

Bài này nhỏ — 95.949 tham số, 16,9 TFLOP cho cả 40 epoch — nên nó bị chặn bởi overhead
mỗi step chứ không bởi sức tính của GPU. Đừng ngạc nhiên nếu T4 không nhanh hơn một card
rời tầm thấp bao nhiêu.

In [ ]:
import subprocess
import sys
import time

# subprocess chu KHONG phai `!`: mot lenh `!` hong van de notebook chay tiep, va
# cell doc ket qua phia duoi se nem FileNotFoundError ve metrics.json - tuc bao
# sai cho. O day exit code khac 0 la dung ngay tai day, kem log that.
CMD = [
    sys.executable, "-m", "masoi_training.train_bc",
    "--data", DATA,
    "--out", "/content/model-v001",
    "--epochs", "40",
    "--batch-size", "512",
    "--lr", "1e-3",
    "--hidden", "128",
    "--seed", "12345",
    "--model-id", "policy-v001",
]

!rm -rf /content/model-v001
start = time.time()
done = subprocess.run(CMD, cwd="/content/bc")
print(f"\ntong {time.time() - start:.0f}s  | exit {done.returncode}")
if done.returncode != 0:
    raise SystemExit(
        f"train_bc hong (exit {done.returncode}). Doc log ngay phia tren.\n"
        "Het RAM thi Colab giet tien trinh va log co the trong: xem muc 'Neu het RAM' o cuoi notebook."
    )


## 5. Đọc kết quả

Số quyết định được là **`metrics.test.agreementTieAware`** — model có chọn một nước HOÀ
ĐỈNH với teacher không, trên những ván nó chưa từng thấy.

Không phải `agreement`. Thước cũ đó chấm oan mọi nước hoà điểm mà teacher phá hoà bằng
id thô — thứ §9 cố tình giấu khỏi observation, nên model không có đường nào học được.
Nó vẫn được in ra để so với bảng số cũ, không phải để kết luận.

Và đừng kết luận bằng loss. §17 nói rõ: chưa tái lập được bot thì chưa được sang RL.
Loss giảm mà agreement không tăng nghĩa là model đang học phân bố của lớp đông nhất,
không phải học chơi.

In [ ]:
import json
import pathlib

path = pathlib.Path("/content/model-v001/metrics.json")
if not path.exists():
    raise SystemExit(
        f"Chua co {path} - tuc cell 4 (train) CHUA chay xong hoac da hong.\n"
        "Quay lai cell 4, doc dong 'exit ...' cuoi cell. Dung sua cell nay."
    )
report = json.loads(path.read_text())
m = report["metrics"]

print(f"{'':<11} {'tieAware':>9} {'agreement':>10} {'top-2':>8}")
for name in ("train", "validation", "test"):
    e = m[name]
    print(f"{name:<11} {e.get('agreementTieAware'):>9} {e.get('agreement'):>10} {e.get('top2Agreement'):>8}")

# Chẩn đoán trên CHÍNH thước dùng để kết luận. Đọc khoảng cách train-val bằng
# `agreement` trong khi kết luận bằng `agreementTieAware` là trộn hai thang.
gap = m["train"]["agreementTieAware"] - m["validation"]["agreementTieAware"]
print(f"\ntrain - val = {gap:+.4f}  ->", "OVERFIT" if gap > 0.05 else "chua overfit")
print("epoch tot nhat:", report["bestEpoch"], "/", report["trainingConfig"]["epochs"])

print("\nTheo loai quyet dinh (thap nhat truoc):")
for kind, score in sorted(m["test"].get("agreementByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

print("\nTheo vai (vai nao model bam kem nhat):")
for role, score in sorted(m["test"].get("agreementByRole", {}).items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")

In [ ]:
import matplotlib.pyplot as plt

history = report["history"]
epochs = [row["epoch"] for row in history]

figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row["trainLoss"] for row in history], label="train loss")
left.set_xlabel("epoch")
left.set_ylabel("loss")

right = left.twinx()
right.plot(
    epochs,
    [r["val_agreementTieAware"] if r.get("val_agreementTieAware") is not None else r.get("val_agreement") for r in history],
    color="tab:orange",
    label="val agreement",
)
right.set_ylabel("agreement")

figure.legend(loc="upper right")
plt.title("Behavior cloning: loss giam KHONG du, agreement moi la thu phai tang")
plt.show()

## 6. Lưu model về Drive

Runtime TypeScript đọc **`model.weights.json`** (định dạng `masoi-mlp-1`), KHÔNG đọc
`model.onnx`: `game-engine` phải thuần và mọi quyết định của bot là đồng bộ, trong khi
`onnxruntime-node` chỉ có API bất đồng bộ (xem `masoi_training/export.py`). File `.onnx`
chỉ để xem cấu trúc hoặc mang sang công cụ khác.

`metrics.json` mang `modelId`, `gitCommit`, `datasetVersion` và `trainingSeed` — một
model không truy ngược được về dataset và commit đã sinh ra nó là một model không tái
lập được (§46), nên giữ nó cùng thư mục với `model.weights.json`.

Ở máy bạn, đặt cả thư mục vào `.tmp/bc/model-v001/` rồi benchmark trước khi nghĩ tới
production:

```bash
npm run ai:benchmark -- --model .tmp/bc/model-v001/model.weights.json --setups baseline,village,wolves,all,teacher
```

KHÔNG ghi đè `apps/server/assets/models/` — đó là model đang đóng gói vào image.


In [ ]:
!mkdir -p /content/drive/MyDrive/masoi-models
!cp -r /content/model-v001 /content/drive/MyDrive/masoi-models/
!ls -lh /content/drive/MyDrive/masoi-models/model-v001

# Hoac tai thang ve may (model nho, vai MB):
# from google.colab import files
# !cd /content && zip -qr model-v001.zip model-v001
# files.download("/content/model-v001.zip")

## Cách đọc con số

Đọc `test.agreementTieAware`, và đọc nó so với **trần của tập**, không so với 1,0.

Trần đó là con số `npm run ai:validate-dataset` in ra ở dòng "trần độ khớp (§17)". Với
dataset 10.002 ván sinh ngày 2026-09-16 nó là **0,984**. Phần thiếu là những nước mà
chính teacher phá hoà bằng thông tin không có trong observation; không policy nào học
được phần đó.

| tieAware / trần | Nghĩa là | Làm gì |
|---|---|---|
| < 0,35 | Chưa học được gì đáng kể | Xem mục dưới |
| 0,35 – 0,70 | Học được xu hướng, chưa tái lập bot | Tăng `--epochs` / `--hidden` |
| > 0,75 | Tái lập baseline khá tốt | Đủ điều kiện §17 để tính tới RL |

### Nếu agreement chững

Nhìn `agreementByDecision` **trước** khi đụng vào model. `SPEECH` không có nhãn trong
không gian hành động này — 862.075 / 1.928.813 dòng là "không nhãn" đúng vì lý do đó —
nên nếu một loại quyết định nào đó kéo trung bình xuống thì đó là chỗ để soi, không phải
`--hidden`.

Tăng `--hidden` chỉ giúp khi train ≈ val và cả hai đều thấp, tức underfit. Nếu train ≫
val thì tăng model là làm nặng thêm đúng cái bệnh đang có.

Đừng chọn seed cho ra kết quả đẹp nhất. Đổi `--seed` là để kiểm model có ổn định không;
nếu hai seed lệch nhau nhiều thì con số nào cũng chưa kết luận được gì.

## Neu het RAM (exit -9)

`exit -9` la SIGKILL: Linux giet tien trinh vi het RAM. Log thuong TRONG, nen no khong
tu noi ra nguyen nhan.

`masoi_training.data.load()` nap TOAN BO tensor vao RAM, roi `split()` cat ra ba ban sao.
Do tren dataset 10.002 van:

| | goi day du | goi slim |
|---|---|---|
| full | 3,08 GB | 2,21 GB |
| + train / val / test | 2,12 + 0,48 + 0,48 | 1,52 + 0,34 + 0,34 |
| **mot tien trinh** | **6,16 GB** | **4,41 GB** |

Colab free co khoang 12,7 GB. Mot tien trinh thi vua. **Hai** thi khong - va do dung la
cai bay: neu cell kiem du lieu giu bien `data` song, notebook dang om 3,08 GB trong khi
cell train spawn mot tien trinh con nap them 6,16 GB nua. Cell kiem o tren da duoc sua
de tra RAM lai ngay sau khi kiem xong.

Van het RAM thi bo `scores.f32.bin` - no CHI dung khi `--distill-alpha > 0`, ma mac dinh
la 0:

```powershell
Remove-Item .tmp\bc\enc\scores.f32.bin
Compress-Archive -Path ai-training\masoi_training, .tmp\bc\enc -DestinationPath .tmp\bc\bc-package-slim.zip -Force
```

`data.py` coi file nay la tuy chon nen khong can sua gi them.

Cach thu ba: `Runtime > Change runtime type > High-RAM`, neu tai khoan cua ban co.

Do that (RSS cua tien trinh, dataset day du): truoc cell kiem 0,03 GB -> trong ham
3,12 GB (dinh 5,25 GB luc cat split) -> sau khi ham tra ve 0,03 GB. Tuc RAM duoc tra
lai tron ven truoc khi cell train chay.
